In [0]:
#%pip install xgboost

import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.sql import functions as F
from xgboost.spark import SparkXGBClassifier

In [0]:
df_model = spark.table("delitos.gold.delitos_cdmx_features")
df_model.printSchema()

In [0]:
# Columnas que necesitan encoding
categoricas = ["AlcaldiaHechos"]

# Columnas numéricas que van directo al vector
numericas = [
    "hora_del_dia", "dia_semana", "dia_mes", "trimestre",
    "mes_hechos_num", "longitud", "latitud", "dias_para_registro",
    "conteo_alcaldia_hora", "conteo_alcaldia_dia", "conteo_alcaldia_mes"
]

In [0]:
df_model.groupBy("categoria_delito") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(truncate=False)

In [0]:
total = df_model.count()
num_clases = df_model.select("categoria_delito").distinct().count()

pesos = df_model.groupBy("categoria_delito") \
    .count() \
    .withColumn("weight", F.lit(total) / (F.lit(num_clases) * F.col("count"))) \
    .select("categoria_delito", "weight")

pesos.orderBy("weight").show(truncate=False)

In [0]:
df_model = df_model.join(pesos, on="categoria_delito", how="left")

In [0]:
df_model.select("categoria_delito", "weight").show(5, truncate=False)
df_model.count()

In [0]:
# 1. Indexar el target
indexer_target = StringIndexer(
    inputCol="categoria_delito",
    outputCol="label"
)
# 2. Indexar AlcaldiaHechos
indexer_alcaldia = StringIndexer(
    inputCol="AlcaldiaHechos",
    outputCol="alcaldia_idx"
)

# 3. OneHotEncoder para alcaldia
encoder = OneHotEncoder(
    inputCol="alcaldia_idx",
    outputCol="alcaldia_vec"
)




In [0]:
total = df_model.count()
num_clases = 16

pesos_alcaldia = df_model.groupBy("AlcaldiaHechos") \
    .count() \
    .withColumn("weight", F.lit(total) / (F.lit(num_clases) * F.col("count"))) \
    .select("AlcaldiaHechos", "weight")

pesos_alcaldia.orderBy("weight").show(truncate=False)

In [0]:
# Indexar el nuevo target
indexer_target2 = StringIndexer(
    inputCol="AlcaldiaHechos",
    outputCol="label"
)

# Indexar categoria_delito (ahora es un feature)
indexer_categoria = StringIndexer(
    inputCol="categoria_delito",
    outputCol="categoria_idx"
)

# OneHotEncoder para categoria
encoder_categoria = OneHotEncoder(
    inputCol="categoria_idx",
    outputCol="categoria_vec"
)

# VectorAssembler
assembler2 = VectorAssembler(
    inputCols=[
        "categoria_vec", "hora_del_dia", "dia_semana", "dia_mes",
        "trimestre", "mes_hechos_num", "longitud", "latitud",
        "dias_para_registro", "conteo_alcaldia_hora",
        "conteo_alcaldia_dia", "conteo_alcaldia_mes"
    ],
    outputCol="features"
)

# Modelo
rf2 = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    weightCol="weight",
    numTrees=50,
    maxDepth=8,
    seed=42
)

# Pipeline
pipeline2 = Pipeline(stages=[
    indexer_target2,
    indexer_categoria,
    encoder_categoria,
    assembler2,
    rf2
])

In [0]:
pesos_alcaldia = df_model.groupBy("AlcaldiaHechos") \
    .count() \
    .withColumn("weight", F.lit(total) / (F.lit(num_clases) * F.col("count"))) \
    .select("AlcaldiaHechos", "weight")

# Lee de Gold y une los pesos
df_model2 = spark.table("delitos.gold.delitos_cdmx_features") \
    .join(pesos_alcaldia, on="AlcaldiaHechos", how="left")

df_model2.count()

In [0]:
train2, test2 = df_model2.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train2.count()}")
print(f"Test: {test2.count()}")

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# Evaluadores
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)


In [0]:

with mlflow.start_run(run_name="rf_alcaldia_target"):
    
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 8)
    mlflow.log_param("target", "AlcaldiaHechos")
    
    model2 = pipeline2.fit(train2)
    
    predictions2 = model2.transform(test2)
    
    accuracy2 = evaluator_acc.evaluate(predictions2)
    f1_2 = evaluator_f1.evaluate(predictions2)
    
    mlflow.log_metric("accuracy", accuracy2)
    mlflow.log_metric("f1", f1_2)
    mlflow.spark.log_model(model2, "modelo_rf_alcaldia")

print(f"Accuracy: {round(accuracy2 * 100, 2)}%")
print(f"F1 Score: {round(f1_2, 4)}")

In [0]:
# Accuracy en train
predictions_train = model2.transform(train2)
accuracy_train = evaluator_acc.evaluate(predictions_train)
print(f"Accuracy Train: {round(accuracy_train * 100, 2)}%")
print(f"Accuracy Test:  {round(accuracy2 * 100, 2)}%")

In [0]:
from mlflow.models.signature import infer_signature

# Usa predictions2 que ya tiene features y prediction
input_sample = predictions2.select("features").limit(10).toPandas()
output_sample = predictions2.select("prediction").limit(10).toPandas()

signature = infer_signature(input_sample, output_sample)

# Loggea de nuevo con firma
with mlflow.start_run(run_name="rf_alcaldia_con_firma"):
    
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 8)
    mlflow.log_param("target", "AlcaldiaHechos")
    mlflow.log_metric("accuracy", accuracy2)
    mlflow.log_metric("f1", f1_2)
    
    mlflow.spark.log_model(
        model2,
        "modelo_rf_alcaldia",
        signature=signature
    )
    
    run_id = mlflow.active_run().info.run_id

print(f"Run ID: {run_id}")  

In [0]:
model_uri = f"runs:/{run_id}/modelo_rf_alcaldia"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="delitos.gold.delitos_cdmx_alcaldia_predictor"
)

print(f"Modelo registrado: {registered_model.name}")
print(f"Versión: {registered_model.version}")

In [0]:
# Haz predicciones con el modelo que ya está en memoria
nuevos_datos = spark.table("delitos.gold.delitos_cdmx_features").limit(10)
predicciones = model2.transform(nuevos_datos)

predicciones.select(
    "categoria_delito",
    "hora_del_dia",
    "dia_semana",
    "AlcaldiaHechos",
    "prediction"
).show(truncate=False)

In [0]:
%sql
SHOW VOLUMES

In [0]:
import mlflow
from pyspark.sql.functions import struct, col

model_uri = 'runs:/408e895b93fe4ea1a6303385b32426dd/modelo_rf_alcaldia'

# Load model as a Spark UDF. Override result_type if the model does not return double values.
loaded_model = mlflow.pyfunc.spark_udf(spark, model_uri=model_uri)

# Predict on a Spark DataFrame.
#df.withColumn('predictions', loaded_model(struct(*map(col, df.columns))))

In [0]:
# Toma los promedios históricos de cada alcaldía
promedios = df_model2.groupBy("AlcaldiaHechos") \
    .agg(
        F.avg("longitud").alias("longitud"),
        F.avg("latitud").alias("latitud"),
        F.avg("conteo_alcaldia_hora").alias("conteo_alcaldia_hora"),
        F.avg("conteo_alcaldia_dia").alias("conteo_alcaldia_dia"),
        F.avg("conteo_alcaldia_mes").alias("conteo_alcaldia_mes"),
        F.avg("dias_para_registro").alias("dias_para_registro")
    )

# Simula martes a las 11pm con delito más común
escenario = promedios \
    .withColumn("hora_del_dia", F.lit(23)) \
    .withColumn("dia_semana", F.lit(3)) \
    .withColumn("dia_mes", F.lit(15)) \
    .withColumn("trimestre", F.lit(4)) \
    .withColumn("mes_hechos_num", F.lit(11)) \
    .withColumn("categoria_delito", F.lit("DELITO DE BAJO IMPACTO"))

escenario.show(truncate=False)

In [0]:
predicciones_patrullaje = model2.transform(escenario)

from pyspark.ml.feature import IndexToString

converter = IndexToString(
    inputCol="prediction",
    outputCol="alcaldia_predicha",
    labels=model2.stages[0].labels
)

resultado = converter.transform(predicciones_patrullaje)

resultado.select(
    "AlcaldiaHechos",
    "alcaldia_predicha",
    "hora_del_dia",
    "dia_semana"
).show(truncate=False)

In [0]:
# Genera el ranking de alcaldías con mayor demanda de patrullaje
ranking = resultado.groupBy("alcaldia_predicha") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .withColumnRenamed("alcaldia_predicha", "alcaldia_riesgo") \
    .withColumnRenamed("count", "zonas_apuntando_aqui")

ranking.show(truncate=False)

In [0]:
def simular_patrullaje(dia_semana, hora):
    escenario = promedios \
        .withColumn("hora_del_dia", F.lit(hora)) \
        .withColumn("dia_semana", F.lit(dia_semana)) \
        .withColumn("dia_mes", F.lit(15)) \
        .withColumn("trimestre", F.lit(4)) \
        .withColumn("mes_hechos_num", F.lit(11)) \
        .withColumn("categoria_delito", F.lit("DELITO DE BAJO IMPACTO"))
    
    predicciones = model2.transform(escenario)
    resultado = converter.transform(predicciones)
    
    print(f"=== Ranking de patrullaje — Día {dia_semana}, Hora {hora}:00 ===")
    resultado.groupBy("alcaldia_predicha") \
        .count() \
        .orderBy(F.col("count").desc()) \
        .withColumnRenamed("alcaldia_predicha", "Alcaldía") \
        .withColumnRenamed("count", "Prioridad") \
        .show(truncate=False)

# Prueba con diferentes escenarios
simular_patrullaje(dia_semana=6, hora=2)   # sábado a las 2am
simular_patrullaje(dia_semana=2, hora=8)   # lunes a las 8am
simular_patrullaje(dia_semana=5, hora=18)  # viernes a las 6pm

In [0]:
# Conteos reales por alcaldía Y hora específica
conteos_hora = df_model2.groupBy("AlcaldiaHechos", "hora_del_dia") \
    .count() \
    .withColumnRenamed("count", "conteo_alcaldia_hora")

# Conteos reales por alcaldía Y día específico
conteos_dia = df_model2.groupBy("AlcaldiaHechos", "dia_semana") \
    .count() \
    .withColumnRenamed("count", "conteo_alcaldia_dia")

# Conteos por alcaldía Y mes
conteos_mes = df_model2.groupBy("AlcaldiaHechos", "mes_hechos_num") \
    .count() \
    .withColumnRenamed("count", "conteo_alcaldia_mes")

In [0]:
def simular_patrullaje(dia_semana, hora, mes=11):
    
    # Filtra conteos para la hora y día específicos
    c_hora = conteos_hora.filter(F.col("hora_del_dia") == hora)
    c_dia = conteos_dia.filter(F.col("dia_semana") == dia_semana)
    c_mes = conteos_mes.filter(F.col("mes_hechos_num") == mes)
    
    # Une todo por alcaldía
    escenario = promedios \
        .drop("conteo_alcaldia_hora", "conteo_alcaldia_dia", "conteo_alcaldia_mes") \
        .join(c_hora.select("AlcaldiaHechos", "conteo_alcaldia_hora"), on="AlcaldiaHechos") \
        .join(c_dia.select("AlcaldiaHechos", "conteo_alcaldia_dia"), on="AlcaldiaHechos") \
        .join(c_mes.select("AlcaldiaHechos", "conteo_alcaldia_mes"), on="AlcaldiaHechos") \
        .withColumn("hora_del_dia", F.lit(hora)) \
        .withColumn("dia_semana", F.lit(dia_semana)) \
        .withColumn("dia_mes", F.lit(15)) \
        .withColumn("trimestre", F.lit(4)) \
        .withColumn("mes_hechos_num", F.lit(mes)) \
        .withColumn("categoria_delito", F.lit("DELITO DE BAJO IMPACTO"))
    
    predicciones = model2.transform(escenario)
    resultado = converter.transform(predicciones)
    
    print(f"=== Ranking de patrullaje — Día {dia_semana}, Hora {hora}:00 ===")
    resultado.groupBy("alcaldia_predicha") \
        .count() \
        .orderBy(F.col("count").desc()) \
        .withColumnRenamed("alcaldia_predicha", "Alcaldía") \
        .withColumnRenamed("count", "Prioridad") \
        .show(truncate=False)

# Prueba de nuevo
simular_patrullaje(dia_semana=6, hora=2)
simular_patrullaje(dia_semana=2, hora=8)
simular_patrullaje(dia_semana=5, hora=18)